# OneLake Datastore Integration with Azure ML

This notebook demonstrates how to:
1. **Configure OneLake datastore** in Azure ML workspace
2. **Upload files to OneLake** using ABFS paths and Azure Data Lake Storage SDK
3. **Register data assets** from OneLake for use in Azure ML pipelines
4. **Access OneLake data** through Azure ML datastore paths

## Prerequisites
- Microsoft Fabric workspace with a Lakehouse
- Azure ML workspace
- Proper permissions for both Fabric and Azure ML
- Environment variables configured in `.env` file

In [ ]:
# Import required libraries
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential, ManagedIdentityCredential
from azure.ai.ml.entities import OneLakeDatastore, OneLakeArtifact

from azure.ai.ml import MLClient, Input, Output
from azure.ai.ml.dsl import pipeline
from azure.ai.ml import load_component
from azure.ai.ml.entities import (
    Environment,
    BuildContext,
    Model,
    ManagedOnlineEndpoint,
    ManagedOnlineDeployment,
    CodeConfiguration,
)
from azure.ai.ml.constants import AssetTypes
import time, datetime, os
from dotenv import load_dotenv

load_dotenv(override=True)

SUBSCRIPTION_ID = os.environ["SUBSCRIPTION_ID"]
RESOURCE_GROUP = os.environ["DEV_RESOURCE_GROUP"]
AML_WORKSPACE_NAME = os.environ["DEV_WORKSPACE_NAME"]
EXTERNAL_REGISTRY_NAME = os.environ["DEV_REGISTRY_NAME"]
ONE_LAKE_ARTIFACT = os.environ["ONE_LAKE_ARTIFACT"]+"/Files"
ONE_LAKE_ENDPOINT = os.environ["ONE_LAKE_ENDPOINT"]
ONE_LAKE_WORKSPACE_NAME = os.environ["ONE_LAKE_WORKSPACE_NAME"]

## 1. Setup and Configuration

Import required libraries and load environment variables.

In [ ]:
# Create Azure ML client
ml_client = MLClient(
    DefaultAzureCredential(), SUBSCRIPTION_ID, RESOURCE_GROUP, AML_WORKSPACE_NAME
)

## 2. Create OneLake Datastore in Azure ML

Register a OneLake datastore that points to your Microsoft Fabric Lakehouse.

In [ ]:
# Create and register OneLake datastore
store = OneLakeDatastore(
    name="onelake_example_id",
    description="Datastore pointing to a Microsoft Fabric Lakehouse artifact.",
    one_lake_workspace_name=ONE_LAKE_WORKSPACE_NAME,
    endpoint=ONE_LAKE_ENDPOINT,
    artifact=OneLakeArtifact(
        name=ONE_LAKE_ARTIFACT,  # Format: {lakehouse_name}.Lakehouse/Files
        type="lake_house"
    )
)

# Register the datastore with Azure ML workspace
ml_client.create_or_update(store)
print(f"✓ OneLake datastore '{store.name}' registered successfully")

In [ ]:
# Extract configuration details from the registered datastore
artifact_name = store.artifact.name
artifact_type = store.artifact.type
workspace_name = store.one_lake_workspace_name
endpoint = store.endpoint

# Construct the full artifact URL (for reference)
artifact_url = f"https://{endpoint}/{workspace_name}/{artifact_name}"

print(f"OneLake Workspace: {workspace_name}")
print(f"Artifact: {artifact_name}")
print(f"Full URL: {artifact_url}")

## 3. Extract Datastore Configuration

Get the configuration details from the registered datastore for later use.

## 4. Upload Files to OneLake

Upload files directly to OneLake using ABFS paths and the Azure Data Lake Storage SDK.

**Note**: OneLake datastores in Azure ML are **read-only**. To write data to OneLake, use the ABFS protocol with `azure-storage-file-datalake` SDK.

In [ ]:
# Upload a file to OneLake using ABFS path and Azure Data Lake Storage SDK
from azure.storage.filedatalake import DataLakeServiceClient

# Configure OneLake endpoint using existing datastore variables
onelake_endpoint = f"https://{endpoint}"
file_system = workspace_name  # In OneLake, the workspace is the file system
lakehouse_path = artifact_name.replace("/Files", "")  # Remove /Files suffix

# Create DataLakeServiceClient for OneLake
service_client = DataLakeServiceClient(
    account_url=onelake_endpoint,
    credential=DefaultAzureCredential()
)

# Get file system (workspace) and directory client
file_system_client = service_client.get_file_system_client(file_system)
directory_client = file_system_client.get_directory_client(
    f"{lakehouse_path}/Files/uploaded_data"
)

# Create the directory if it doesn't exist
try:
    directory_client.create_directory()
    print(f"✓ Created directory: {lakehouse_path}/Files/uploaded_data")
except Exception as e:
    print(f"ℹ Directory already exists or error: {e}")

# Upload a local file to OneLake
local_file_path = "../../data/taxi-data/raw/greenTaxiData.csv"
remote_file_name = "greenTaxiData_uploaded.csv"

file_client = directory_client.get_file_client(remote_file_name)

with open(local_file_path, "rb") as data:
    file_client.upload_data(data, overwrite=True)
    
print(f"✓ Uploaded: {local_file_path} → {remote_file_name}")
print(f"\nABFS path: abfss://{workspace_name}@{endpoint}/{artifact_name}/uploaded_data/{remote_file_name}")
print(f"Datastore path: azureml://datastores/onelake_example_id/paths/uploaded_data/{remote_file_name}")

In [ ]:
# Register the uploaded file as an Azure ML data asset
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

# Reference the file we uploaded via the OneLake datastore
uploaded_data = Data(
    name="onelake_uploaded_green_taxi",
    version="1",
    description="Green taxi data uploaded via ABFS to OneLake",
    type=AssetTypes.URI_FILE,
    path="azureml://datastores/onelake_example_id/paths/uploaded_data/greenTaxiData_uploaded.csv"
)

# Register the data asset in Azure ML
ml_client.data.create_or_update(uploaded_data)

print(f"✓ Data asset registered: {uploaded_data.name}")
print(f"✓ Use in pipelines: azureml:{uploaded_data.name}:{uploaded_data.version}")
print(f"\nℹ This data asset points to OneLake and can be used as input in Azure ML pipelines")

Data asset registered: onelake_uploaded_green_taxi
You can now use this in pipelines with: azureml:onelake_uploaded_green_taxi:1


## 5. Register OneLake Data as Azure ML Asset

Register files from OneLake as data assets in Azure ML for use in pipelines and jobs.

## Summary

This notebook demonstrated:

✅ **OneLake Datastore Registration** - Connected Azure ML to Microsoft Fabric Lakehouse  
✅ **File Upload to OneLake** - Used ABFS protocol and Azure Data Lake Storage SDK to write data  
✅ **Data Asset Registration** - Registered OneLake files as Azure ML data assets  
✅ **Pipeline Integration** - Created reusable data assets that can be consumed in Azure ML pipelines

### Key Takeaways

- **OneLake datastores are read-only in Azure ML** - Use ABFS + `azure-storage-file-datalake` SDK for uploads
- **Use datastore paths for reading** - Reference OneLake data via `azureml://datastores/{name}/paths/{path}`
- **Register data assets** - Make OneLake data discoverable and versioned in Azure ML
- **Pipeline outputs cannot write to OneLake directly** - Outputs must go to Azure Blob Storage, then copy to OneLake if needed

### Next Steps

- Run pipelines that consume OneLake data as inputs
- Set up automated copy processes from Azure Blob Storage to OneLake for pipeline outputs
- Implement data lineage tracking between Azure ML and Microsoft Fabric